In [26]:

import torch
import torch.nn as nn
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 数据采集

In [27]:
train_path = './ImageNet/train'
valid_path = './ImageNet/valid'
train_transform = v2.Compose([
    v2.RandomResizedCrop((224, 224)),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224,0.225]),
])
valid_transform = v2.Compose([
    v2.Resize((256, 256)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224,0.225]),
])

In [28]:
train_dataset  = ImageFolder(root=train_path,transform=train_transform)
valid_dataset  = ImageFolder(root=valid_path,transform=valid_transform)

In [29]:
train_data_loader = DataLoader(train_dataset, batch_size=32, num_workers=4, shuffle = True, drop_last=True)
valid_data_loader = DataLoader(valid_dataset, batch_size=32, num_workers=4, shuffle = False, drop_last=True)

# 残差神经网络

In [30]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, hidden_channels):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        self.out_channels = out_channels
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, kernel_size=1, padding=0),
            nn.BatchNorm2d(hidden_channels),
            nn.SiLU(),
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.SiLU(),
            nn.Conv2d(hidden_channels, out_channels, kernel_size=1, padding=0),
        )
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, padding=0),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        output = self.cnn(x)
        input = x
        if self.in_channels != self.out_channels:
            input = self.shortcut(x)
        output += input
        output = torch.nn.SiLU()(output)
        return output

In [31]:
class Resnet(nn.Module):
    def __init__(self, block=ResBlock):
        super().__init__()
        self.block = block
        base_config = [
            # 输入通道, 输出通道，隐藏通道, block数
            (64, 256, 64, 3),  # Stage 1
            (256, 512, 128, 4),  # Stage 2
            (512, 1024, 256, 6),  # Stage 3
            (1024, 2048, 512, 3),  # Stage 4
        ]

        self.layers = [nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2, padding=3),
              nn.SiLU(),
              nn.MaxPool2d(kernel_size=3, stride=2, padding=1)]
    
        for in_channels, out_channels, hidden_channels, num_blocks in base_config:
            self.layers.append(nn.Sequential(
                *[block(in_channels if i == 0 else out_channels, out_channels, hidden_channels) for i in range(num_blocks)]
        ))

        self.layers.append(nn.AdaptiveAvgPool2d((1, 1)))
        self.layers.append(nn.Flatten())
        self.layers.append(nn.Dropout(0.2))
        self.layers.append(nn.Linear(2048, 20))
        self.model = nn.Sequential(*self.layers)


    def forward(self, x):
        x = self.model(x)
        return x


In [32]:
# class ResNet(nn.Module):
#     def __init__(self, in_channels, out_channels, hidden_channels, res_channels, num_res):
#         super().__init__()
#         self.in_channels = in_channels
#         self.out_channels = out_channels
#         self.res_channels = res_channels
#         self.hidden_channels = hidden_channels
#         self.num_res = num_res
#         self.encoder = nn.Sequential(
#             nn.Conv2d(in_channels, res_channels, kernel_size=7, stride=2, padding=3),
#             nn.SiLU(),
#             nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
#         )
#         self.decoder = nn.Conv2d(res_channels, out_channels, kernel_size=3, padding=1)
#         self.hid_res_layers = nn.Sequential(
#             *[ResBlock(res_channels, res_channels, hidden_channels) for _ in range(num_res)]
#         )

#     def forward(self, x):
#         y = self.encoder(x)
#         y = self.hid_res_layers(y)
#         y = self.decoder(y)
#         return y


# 训练

In [33]:
def train(model=None, data_loader=train_data_loader, lr=0.001):
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fun = nn.CrossEntropyLoss()
    total_correct = 0
    total_loss = 0
    train_counter = 0
    model.train()
    batch_size = data_loader.batch_size
    for data, label in tqdm(data_loader, desc=f"Training"):
        data = data.to(device)
        label = label.to(device)
        optim.zero_grad()
        pred = model(data)
        loss = loss_fun(pred, label)
        loss.backward()
        optim.step()
        pred_labels = pred.argmax(dim=1)
        correct = (pred_labels == label).sum().item()
        total_correct += correct
        total_loss += loss.item()
        train_counter += 1
        if train_counter % 1000 == 0:
            print(f"Step: {train_counter}, Loss: {total_loss / train_counter}, Accuracy: {total_correct / (train_counter * batch_size)}")
    print(f"Train Accuracy: {total_correct / (len(data_loader) * batch_size)}")
    return None

# 验证

In [34]:
def evaluate(model=None, data_loader=valid_data_loader):
    model.eval()
    
    total_correct = 0
    val_counter = 0
    total_loss = 0
    batch_size = data_loader.batch_size
    loss_fun = nn.CrossEntropyLoss()
    with torch.no_grad():
        for data, label in tqdm(data_loader, desc="Evaluating"):
            data = data.to(device)
            label = label.to(device)
            pred = model(data)
            loss = loss_fun(pred, label)
            total_loss += loss.item()
            val_counter += 1
            pred_labels = pred.argmax(dim=1)
            correct = (pred_labels == label).sum().item()
            total_correct += correct
            if val_counter % 1000 == 0:
                print(f"Step: {val_counter}, Loss: {total_loss / val_counter}, Accuracy: {total_correct / (val_counter * batch_size)}")
    print(f"Validation Accuracy: {total_correct / (len(data_loader) * batch_size)}")
    return None

In [35]:
def export_model(model, path="./resnet.pth"):
    torch.save(model.state_dict(), path)

In [36]:
def main():
    net = Resnet()
    net = net.to(device)
    for epoch in range(10):
        print(f"Epoch {epoch + 1}/10:")
        train(model=net)
        evaluate(model=net)
    print("Training complete. Exporting model...")
    export_model(net)
    print("Model exported successfully.")
    

In [37]:
main()

Epoch 1/10:


Training:   0%|          | 0/761 [00:18<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 50.00 MiB. GPU 0 has a total capacity of 7.96 GiB of which 0 bytes is free. Of the allocated memory 21.90 GiB is allocated by PyTorch, and 144.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)